<a href="https://colab.research.google.com/github/Avivek121/Ai-Training-Project-Loan-Prediction-System/blob/main/loan_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
df = pd.read_csv('/content/loan_sanction_train.csv')

In [15]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [16]:
df.shape

(614, 13)

In [17]:
df.isnull().sum().sum()

np.int64(149)

In [18]:
df = df.dropna()

In [19]:
df.isnull().sum().sum()

np.int64(0)

In [20]:
df = df.dropna(axis=1)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 480 entries, 1 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            480 non-null    object 
 1   Gender             480 non-null    object 
 2   Married            480 non-null    object 
 3   Dependents         480 non-null    object 
 4   Education          480 non-null    object 
 5   Self_Employed      480 non-null    object 
 6   ApplicantIncome    480 non-null    int64  
 7   CoapplicantIncome  480 non-null    float64
 8   LoanAmount         480 non-null    float64
 9   Loan_Amount_Term   480 non-null    float64
 10  Credit_History     480 non-null    float64
 11  Property_Area      480 non-null    object 
 12  Loan_Status        480 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 52.5+ KB


In [23]:
x = df.drop('Married', axis=1)
y = df['Married']
print("Shape of x", x.shape)
print("Shape of y", y.shape)

Shape of x (480, 12)
Shape of y (480,)


In [24]:
from sklearn.model_selection import train_test_split
from numpy.random import random_sample
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=51)

In [25]:
print("shape of x_train",x_train.shape)
print("shape of x_test",x_test.shape)

shape of x_train (384, 12)
shape of x_test (96, 12)


In [26]:
print("shape of y_train",y_train.shape)
print("shape of y_test",y_test.shape)

shape of y_train (384,)
shape of y_test (96,)


In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# Drop 'Loan_ID' first from x_train and x_test as it's an identifier and not a feature for scaling
x_train_processed = x_train.drop('Loan_ID', axis=1)
x_test_processed = x_test.drop('Loan_ID', axis=1)

# Handle 'Dependents' column: replace '3+' with '3' and convert to integer
x_train_processed['Dependents'] = x_train_processed['Dependents'].replace('3+', '3').astype(int)
x_test_processed['Dependents'] = x_test_processed['Dependents'].replace('3+', '3').astype(int)

# Identify numerical and categorical columns after processing 'Dependents'
numerical_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Dependents']
categorical_cols = ['Gender', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']

# Create a column transformer for preprocessing
# One-hot encode categorical features and scale numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough' # Keep other columns if any, though none expected after this setup
)

# Apply the preprocessor to x_train and x_test
x_train = preprocessor.fit_transform(x_train_processed)
x_test = preprocessor.transform(x_test_processed)

print("Shape of x_train after preprocessing:", x_train.shape)
print("Shape of x_test after preprocessing:", x_test.shape)

Shape of x_train after preprocessing: (384, 17)
Shape of x_test after preprocessing: (96, 17)


In [30]:
from sklearn.linear_model import LinearRegression
y_train_encoded = y_train.map({'Yes': 1, 'No': 0})
y_test_encoded = y_test.map({'Yes': 1, 'No': 0})

lr = LinearRegression()
lr.fit(x_train, y_train_encoded)

LinearRegression()

In [31]:
x_test[0,:]

array([-0.26442471, -0.60102104, -0.41971251,  0.27524759,  0.43457392,
        0.26775113,  1.        ,  0.        ,  1.        ,  0.        ,
        1.        ,  0.        ,  1.        ,  0.        ,  0.        ,
        0.        ,  1.        ])

In [32]:
lr.predict([x_test[0,:]])

array([0.33767207])

In [33]:
lr.predict([x_test[1,:]])

array([0.15260044])

In [34]:
y_test

,Married
238,No
328,Yes
341,No
609,No
397,Yes
...,...
483,Yes
361,Yes
612,Yes
350,Yes


In [36]:
lr.score(x_test, y_test_encoded)

0.26731530226453026

In [37]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [39]:
y_pred_test = lr.predict(x_test)
print("MAE",mean_absolute_error(y_test_encoded,y_pred_test))

MAE 0.3233256217808293


In [41]:
print("MSE",mean_squared_error(y_test_encoded,y_pred_test))

MSE 0.16019527625184155


In [43]:
import numpy as np

print("RMSE",np.sqrt(mean_squared_error(y_test_encoded,y_pred_test)))

RMSE 0.40024402088206334


In [45]:
# R2 Score
from sklearn.metrics import r2_score
print("R2 Score",r2_score(y_test_encoded,y_pred_test))

R2 Score 0.26731530226453026
